<a href="https://colab.research.google.com/github/Sergi-e/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API key set up
from google.colab import userdata
API_KEY = userdata.get('GROQ_API_KEY')

# # TODO: set API_KEY using ONE of the methods above.
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

In [3]:
# Part 1.1 — My first API call

# TODO: helper function I'll reuse for the whole lab
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [4]:
# TODO: call it once with a simple question and print the answer
answer = ask_llm("What is a good name for a savings product for market traders in Accra?")
print(answer)

For a savings product targeting market traders in Accra, you'll want a name that resonates with the local culture and conveys a sense of security, trust, and growth. Here are some suggestions:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name would immediately connect with market traders.
2. **TradeSafe**: This name emphasizes the idea of keeping savings safe and secure, which is essential for market traders who often deal with cash.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect." This name suggests a product that helps traders gather and grow their savings.
4. **MarketMoen**: "Moen" is a Ghanaian word for "progress" or "growth." This name conveys the idea of helping market traders make progress with their savings.
5. **Adwuma Savings**: "Adwuma" means "work" or "business" in the Akan language. This name highlights the product's focus on supporting market traders' businesses.
6. **Kaya Savings**: "Kaya"

In [5]:
# TODO: print response.usage to see how many tokens the call consumed
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a good name for a savings product for market traders in Accra?"},
    ],
)
print(response.usage)

CompletionUsage(completion_tokens=398, prompt_tokens=57, total_tokens=455, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.134356285, prompt_time=0.002698601, completion_time=1.2850292589999999, total_time=1.28772786)


> **Student Reasoning — Anatomy of a call:**
>
> **1. System vs user roles:** The system role sets the model's identity and rules for the whole conversation, while the user role is the actual request being made in that turn. In my ask_llm() function, the system role tells the model to act as a helpful assistant, and the user role asks the specific question about a savings product name for market traders in Accra.
>
> **2. Tokens and billing:** A token is roughly a piece of a word, sometimes a whole short word and sometimes just a few characters, that the model reads and generates one at a time. My test call used 57 tokens for the prompt and 398 for the completion, for a total of 455. Providers bill per token instead of per request because the actual computing cost depends on how much text is processed and generated, not on the fact that a request was made. A one word answer and an eight paragraph list, like the one I got back, cost very different amounts to produce, so token based billing reflects that difference fairly.

In [6]:
# Part 1.2 — temperature: the randomness dial
# TODO: ask the same question 5 times at temperature=0.0 and 5 times at temperature=1.2
savings_question = "Suggest a name for a savings product for market traders in Accra."

low_temp_runs = [ask_llm(savings_question, temperature=0.0) for _ in range(5)]
high_temp_runs = [ask_llm(savings_question, temperature=1.2) for _ in range(5)]

In [7]:
# TODO: print all 10 answers, grouped by temperature
print("Temperature 0.0 runs")
for i, ans in enumerate(low_temp_runs, 1):
    print(f"Run {i}: {ans}")

print("\nTemperature 1.2 runs")
for i, ans in enumerate(high_temp_runs, 1):
    print(f"Run {i}: {ans}")

Temperature 0.0 runs
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the id

> **Student Reasoning — Temperature:**
>
> **1. What I observed:** At temperature 0.0, the answers were nearly identical across runs, the same names kept repeating, and two runs came back word for word the same. At temperature 1.2, the answers varied a lot more, new names appeared that never showed up at temperature 0, and even small details like word meanings shifted between runs.
>
> **2. Which temperature fits the loan decision-support system:** Temperature 0.0 is the right choice for that system. A loan officer needs consistent, repeatable output when reviewing an application, not a different summary or recommendation each time the same letter is processed. High temperature is useful for brainstorming, like generating product name ideas, but it works against reliability in a decision-support context, where trust depends on the system giving the same read on the same facts every time.